# Data preparation

**Purpose.** This notebook turns the three raw files (sales, weather, Kiel Week) into **one clean table** that the baseline model and the main model can use directly. It saves the table as `0_DataPreparation/prepared_data.csv`. The raw files are only read, never changed.

The table has **one row per date and product group**:
- **train** rows: 2013-07-01 to 2018-07-31, with the real turnover (`Umsatz`);
- **test** rows: 2018-08-01 to 2019-07-31 (the Kaggle test year), with the same features but **empty turnover**, because those sales are held back by the course.

Every step follows a team decision (D1–D7, see `DECISIONS.md` and canvas block 7). The checks in `checks/test_prepared_data.py` test the saved table; run them with `uv run pytest checks/`.

**What to check in the results:** the row counts after each step, the number of filled weather days, and the holiday list.

In [1]:
import numpy as np
import pandas as pd
import holidays

BASE_URL = "https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/"
OUTPUT = "prepared_data.csv"  # saved next to this notebook
GROUPS = {1: "Bread", 2: "Rolls", 3: "Croissants", 4: "Confectionery", 5: "Cakes", 6: "Seasonal bread"}
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]  # Monday to Sunday, never alphabetical
TEST_START, TEST_END = "2018-08-01", "2019-07-31"

## 1. Load the raw files

We read the three files straight from the course repository. **Check:** the sales file must have 9,334 rows, as documented.

In [2]:
sales = pd.read_csv(BASE_URL + "umsatzdaten_gekuerzt.csv", parse_dates=["Datum"])
weather = pd.read_csv(BASE_URL + "wetter.csv", parse_dates=["Datum"])
kiwo = pd.read_csv(BASE_URL + "kiwo.csv", parse_dates=["Datum"])

assert len(sales) == 9334, "sales file does not have the documented 9,334 rows"
assert not weather["Datum"].duplicated().any(), "weather has a date twice"
print(f"Sales:      {len(sales):>5} rows, {sales.Datum.min().date()} to {sales.Datum.max().date()}")
print(f"Weather:    {len(weather):>5} rows, {weather.Datum.min().date()} to {weather.Datum.max().date()}")
print(f"Kiel Week:  {len(kiwo):>5} days, years {kiwo.Datum.dt.year.min()} to {kiwo.Datum.dt.year.max()}")

Sales:       9334 rows, 2013-07-01 to 2018-07-31
Weather:     2601 rows, 2012-01-01 to 2019-08-01
Kiel Week:     72 days, years 2012 to 2019


## 2. Build the row skeleton: one row per date and product group

**Training rows (decision D1).** We keep exactly the rows that exist in the sales file. A missing combination (a closed day, or a product not offered that day) means *not sold*, not *zero demand*, so we do **not** fill it with 0.

**Test rows (decision D6).** For the Kaggle test year we create one row per day for product groups 1–5, and for seasonal bread (group 6) only in its months, October to January. The turnover stays empty. The real Kaggle IDs are not in the course repository; later we keep only the rows whose `id` Kaggle asks for.

The `id` follows the course format: `YYMMDD` + product group, e.g. `1307011` = 1 July 2013, bread.

In [3]:
train = sales.assign(set="train")

test_days = pd.date_range(TEST_START, TEST_END)
test = pd.DataFrame([(d, g) for d in test_days for g in GROUPS], columns=["Datum", "Warengruppe"])
test = test[(test.Warengruppe != 6) | test.Datum.dt.month.isin([10, 11, 12, 1])]
test = test.assign(Umsatz=np.nan, set="test")
test["id"] = (test.Datum.dt.strftime("%y%m%d") + test.Warengruppe.astype(str)).astype(int)

df = pd.concat([train, test], ignore_index=True)
df["product_group"] = df.Warengruppe.map(GROUPS)

print(df.groupby(["set", "product_group"], sort=False).size().unstack(0))
print(f"\nTotal rows: {len(df)} (train {len(train)}, test {len(test)})")

set             train  test
product_group              
Bread            1819   365
Rolls            1819   365
Croissants       1819   365
Confectionery    1766   365
Cakes            1819   365
Seasonal bread    292   123

Total rows: 11282 (train 9334, test 1948)


## 3. Add Kiel Week

A **left join** adds a column from another table to every row with the same date and keeps all our rows. `kiwo.csv` lists only Kiel Week days, so every other day gets `KielerWoche = 0`. **Check:** the number of rows must not change; if it did, a date appeared twice in the joined table.

In [4]:
rows_before = len(df)
df = df.merge(kiwo, on="Datum", how="left")
df["KielerWoche"] = df["KielerWoche"].fillna(0).astype(int)
assert len(df) == rows_before, "Kiel Week merge changed the number of rows"

print("Kiel Week days per year in our table:")
print(df[df.KielerWoche == 1].drop_duplicates("Datum").groupby(df.Datum.dt.year).size().to_string())

Kiel Week days per year in our table:
Datum
2014    9
2015    9
2016    9
2017    9
2018    9
2019    9


## 4. Add weather and fill the gaps (decision D2)

Some days have no weather row, and cloud cover (`Bewoelkung`) is missing on a few days. We first build a **complete daily weather calendar**, then fill the gaps by **linear interpolation**: a missing value is set between the values of the day before and the day after. A flag `weather_filled = 1` marks every day where at least one value was filled, so we can check later whether these days behave oddly.

Reminder (decision Q10): this is **observed** weather. In real use only a forecast is known, so results with weather are a *best case*.

In [5]:
WEATHER_COLS = ["Temperatur", "Bewoelkung", "Windgeschwindigkeit"]
all_days = pd.DataFrame({"Datum": pd.date_range(df.Datum.min(), df.Datum.max())})
w = all_days.merge(weather, on="Datum", how="left")

missing = w[WEATHER_COLS].isna()
print("Missing values before filling (days from 2013-07-01 to 2019-07-31):")
print(missing.sum().to_string())
w["weather_filled"] = missing.any(axis=1).astype(int)
w[WEATHER_COLS] = w.set_index("Datum")[WEATHER_COLS].interpolate(method="time").values
w["Bewoelkung"] = w["Bewoelkung"].round()  # cloud cover is measured in whole octas
assert not w[WEATHER_COLS].isna().any().any(), "weather still has gaps after filling"

filled_days = w[w.weather_filled == 1].Datum
print(f"\nDays with filled weather: {len(filled_days)} "
      f"(in training period: {(filled_days <= '2018-07-31').sum()}, in test year: {(filled_days >= TEST_START).sum()})")

Missing values before filling (days from 2013-07-01 to 2019-07-31):
Temperatur             16
Bewoelkung             26
Windgeschwindigkeit    16

Days with filled weather: 26 (in training period: 13, in test year: 13)


## 5. Turn the weather code into "precipitation yes/no" (decision D3)

`Wettercode` follows the WMO code for present weather (00–99). Codes **50–99** describe precipitation: drizzle, rain, snow, showers and thunderstorms. Codes below 50 are dry weather (e.g. fog, haze). The code is missing on many days; the dataset documentation says this most likely means *no notable weather*, so a **missing code counts as no precipitation**.

In [6]:
w["precipitation"] = w["Wettercode"].between(50, 99).astype(int)  # NaN -> False -> 0
print(f"Weather code missing on {w.Wettercode.isna().sum()} of {len(w)} days (counted as no precipitation)")
print(f"Days with precipitation: {w.precipitation.sum()} ({w.precipitation.mean():.0%})")

rows_before = len(df)
df = df.merge(w.drop(columns="Wettercode"), on="Datum", how="left")
assert len(df) == rows_before, "weather merge changed the number of rows"

Weather code missing on 534 of 2222 days (counted as no precipitation)
Days with precipitation: 781 (35%)


## 6. Calendar features (decision D5)

- `weekday` (name) and `weekday_num` (0 = Monday … 6 = Sunday), ordered **Monday to Sunday**;
- `month` (1–12);
- `seasonal_bread_season`: 1 from October to January, the months in which seasonal bread is sold. It is a fixed calendar rule, so it is **known in advance**. The real season start varies by year (24 Oct to 3 Nov, see `FINDINGS.md`).

In [7]:
df["weekday"] = pd.Categorical(df.Datum.dt.day_name(), categories=WEEKDAYS, ordered=True)
df["weekday_num"] = df.Datum.dt.dayofweek
df["month"] = df.Datum.dt.month
df["seasonal_bread_season"] = df.month.isin([10, 11, 12, 1]).astype(int)

print(df.drop_duplicates("Datum").groupby("weekday", observed=True).size().rename("days").to_string())

weekday
Monday       310
Tuesday      314
Wednesday    316
Thursday     311
Friday       308
Saturday     313
Sunday       312


## 7. Holidays (decision D4)

Public holidays for **Schleswig-Holstein** come from the Python package `holidays` (additional data source, listed in `AGENTS.md`). We add:
- `is_holiday`: the day is a public holiday;
- `day_before_holiday`: the next day is a public holiday (people may shop ahead);
- `christmas_eve` (24 Dec) and `new_years_eve` (31 Dec): not public holidays, but they have very large effects in our findings (e.g. cakes on 31 Dec about 7× a normal day).

School holidays are **not** included yet (they need another source).

In [8]:
years = range(df.Datum.dt.year.min(), df.Datum.dt.year.max() + 2)  # +1 year so "day before" works on 31 Dec
sh_holidays = holidays.Germany(subdiv="SH", years=years)
holiday_dates = pd.to_datetime(list(sh_holidays.keys()))

df["is_holiday"] = df.Datum.isin(holiday_dates).astype(int)
df["day_before_holiday"] = (df.Datum + pd.Timedelta(days=1)).isin(holiday_dates).astype(int)
df["christmas_eve"] = ((df.Datum.dt.month == 12) & (df.Datum.dt.day == 24)).astype(int)
df["new_years_eve"] = ((df.Datum.dt.month == 12) & (df.Datum.dt.day == 31)).astype(int)

in_range = sorted(d for d in sh_holidays if df.Datum.min().date() <= d <= df.Datum.max().date())
print(f"Public holidays between {df.Datum.min().date()} and {df.Datum.max().date()}: {len(in_range)}")
print(", ".join(f"{d} {sh_holidays[d]}" for d in in_range[:12]), "...")

Public holidays between 2013-07-01 and 2019-07-31: 56
2013-10-03 Tag der Deutschen Einheit, 2013-12-25 Erster Weihnachtstag, 2013-12-26 Zweiter Weihnachtstag, 2014-01-01 Neujahr, 2014-04-18 Karfreitag, 2014-04-21 Ostermontag, 2014-05-01 Erster Mai, 2014-05-29 Christi Himmelfahrt, 2014-06-09 Pfingstmontag, 2014-10-03 Tag der Deutschen Einheit, 2014-12-25 Erster Weihnachtstag, 2014-12-26 Zweiter Weihnachtstag ...


**Plausibility check:** on a public holiday the shop is often closed, so many holidays should have *no* training rows. We count, for the training period, how many public holidays had any sales.

In [9]:
train_days = set(df.loc[df.set == "train", "Datum"])
hol_train = [d for d in holiday_dates if pd.Timestamp("2013-07-01") <= d <= pd.Timestamp("2018-07-31")]
open_hol = sorted(d for d in hol_train if d in train_days)
print(f"Public holidays in training period: {len(hol_train)}, with sales: {len(open_hol)}, without sales: {len(hol_train) - len(open_hol)}")
print("Holidays with sales:", ", ".join(f"{d.date()} {sh_holidays[d.date()]}" for d in open_hol))

Public holidays in training period: 46, with sales: 20, without sales: 26
Holidays with sales: 2013-10-03 Tag der Deutschen Einheit, 2014-04-21 Ostermontag, 2014-05-29 Christi Himmelfahrt, 2014-06-09 Pfingstmontag, 2014-10-03 Tag der Deutschen Einheit, 2015-04-06 Ostermontag, 2015-05-14 Christi Himmelfahrt, 2015-05-25 Pfingstmontag, 2015-10-03 Tag der Deutschen Einheit, 2016-03-28 Ostermontag, 2016-05-01 Erster Mai, 2016-05-05 Christi Himmelfahrt, 2016-05-16 Pfingstmontag, 2016-10-03 Tag der Deutschen Einheit, 2017-04-17 Ostermontag, 2017-05-25 Christi Himmelfahrt, 2017-06-05 Pfingstmontag, 2017-10-03 Tag der Deutschen Einheit, 2018-04-02 Ostermontag, 2018-05-10 Christi Himmelfahrt


## 8. Temperature bands (decision D5)

For the course bar charts we turn the temperature (a number) into three **bands** (categories): **cold** below 10 °C, **mild** 10 to below 20 °C, **warm** 20 °C and above. The cut points are a draft for the team to check.

In [10]:
df["temp_band"] = pd.cut(df.Temperatur, bins=[-np.inf, 10, 20, np.inf], right=False,
                         labels=["cold (<10 °C)", "mild (10-20 °C)", "warm (≥20 °C)"])
print(df.drop_duplicates("Datum").temp_band.value_counts(sort=False).rename("days").to_string())

temp_band
cold (<10 °C)      900
mild (10-20 °C)    935
warm (≥20 °C)      349


## 9. Final checks and save

Before saving we check the most important rules in the notebook itself; the full set of checks runs with `uv run pytest checks/`.
- exactly one row per date and product group;
- training rows equal the 9,334 raw sales rows; test rows have no turnover;
- no feature has a missing value.

In [11]:
FEATURES = ["weekday", "weekday_num", "month", "KielerWoche", "is_holiday", "day_before_holiday",
            "christmas_eve", "new_years_eve", "seasonal_bread_season",
            "Temperatur", "Bewoelkung", "Windgeschwindigkeit", "weather_filled", "precipitation", "temp_band"]
COLUMNS = ["id", "Datum", "Warengruppe", "product_group", "set", "Umsatz"] + FEATURES

df = df[COLUMNS].sort_values(["Datum", "Warengruppe"]).reset_index(drop=True)

assert not df.duplicated(["Datum", "Warengruppe"]).any(), "a date/product group appears twice"
assert (df.set == "train").sum() == len(sales), "training rows differ from raw sales rows"
assert df.loc[df.set == "test", "Umsatz"].isna().all(), "test rows must have empty turnover"
assert not df[FEATURES].isna().any().any(), "a feature has missing values"

df.to_csv(OUTPUT, index=False, date_format="%Y-%m-%d")
print(f"Saved {OUTPUT}: {len(df)} rows, {len(df.columns)} columns")
df.head()

Saved prepared_data.csv: 11282 rows, 21 columns


,id,Datum,Warengruppe,product_group,set,Umsatz,weekday,weekday_num,month,KielerWoche,...,day_before_holiday,christmas_eve,new_years_eve,seasonal_bread_season,Temperatur,Bewoelkung,Windgeschwindigkeit,weather_filled,precipitation,temp_band
0,1307011,2013-07-01,1,Bread,train,148.828353,Monday,0,7,0,...,0,0,0,0,17.8375,6.0,15.0,0,0,mild (10-20 °C)
1,1307012,2013-07-01,2,Rolls,train,535.856285,Monday,0,7,0,...,0,0,0,0,17.8375,6.0,15.0,0,0,mild (10-20 °C)
2,1307013,2013-07-01,3,Croissants,train,201.198426,Monday,0,7,0,...,0,0,0,0,17.8375,6.0,15.0,0,0,mild (10-20 °C)
3,1307014,2013-07-01,4,Confectionery,train,65.890169,Monday,0,7,0,...,0,0,0,0,17.8375,6.0,15.0,0,0,mild (10-20 °C)
4,1307015,2013-07-01,5,Cakes,train,317.475875,Monday,0,7,0,...,0,0,0,0,17.8375,6.0,15.0,0,0,mild (10-20 °C)


## Summary

- `prepared_data.csv` holds the 9,334 training rows (unchanged turnover) and the test-year rows, with the same features for both.
- Features: weekday, month, Kiel Week, holidays (public holiday, day before, 24 Dec, 31 Dec), seasonal bread season, temperature, cloud cover, wind, precipitation, temperature band, and a flag for filled weather.
- **Not included yet:** recent sales (lag) features (open question on the canvas) and school holidays.
- **Limits:** weather is observed, not forecast (best case); filled weather days are estimates; the test rows will be reduced to the Kaggle IDs once we have them.